## Imports

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import svds
import joblib
try:
    import seaborn as sns
except:
    ! pip install seaborn
    import seaborn as sns
try:
    import optuna
except:
    ! pip install optuna
    import optuna
try:
    import pyarrow
except:
    ! pip install pyarrow
    import pyarrow

optuna.logging.set_verbosity(optuna.logging.WARNING)

## Helper Classes and Functions

In [ ]:
class SVDRecommender:
    """Truncated SVD-based collaborative filtering recommender."""

    def __init__(self, int_k=50):
        self.int_k = int_k

    def fit(self, df_train):
        print(f'Fitting SVD with k={self.int_k}...')

        # create user and item mappings
        list_users = sorted(df_train['user_id'].unique())
        list_items = sorted(df_train['movie_id'].unique())
        self.dict_user_to_idx = {u: i for i, u in enumerate(list_users)}
        self.dict_idx_to_user = {i: u for u, i in self.dict_user_to_idx.items()}
        self.dict_item_to_idx = {m: i for i, m in enumerate(list_items)}
        self.dict_idx_to_item = {i: m for m, i in self.dict_item_to_idx.items()}
        self.int_n_users = len(list_users)
        self.int_n_items = len(list_items)

        # build sparse user-item matrix
        arr_user_idx = df_train['user_id'].map(self.dict_user_to_idx).values
        arr_item_idx = df_train['movie_id'].map(self.dict_item_to_idx).values
        arr_ratings = df_train['rating'].values.astype(np.float64)

        self.flt_global_mean = arr_ratings.mean()
        mat_sparse = csr_matrix(
            (arr_ratings, (arr_user_idx, arr_item_idx)),
            shape=(self.int_n_users, self.int_n_items),
        )

        # compute user and item biases
        mat_dense = mat_sparse.toarray()
        mat_mask = (mat_dense > 0).astype(float)

        # user bias: mean rating per user minus global mean
        arr_user_sum = mat_dense.sum(axis=1)
        arr_user_count = mat_mask.sum(axis=1)
        arr_user_count[arr_user_count == 0] = 1  # avoid division by zero
        self.arr_user_bias = (arr_user_sum / arr_user_count) - self.flt_global_mean

        # item bias: mean rating per item minus global mean
        arr_item_sum = mat_dense.sum(axis=0)
        arr_item_count = mat_mask.sum(axis=0)
        arr_item_count[arr_item_count == 0] = 1
        self.arr_item_bias = (arr_item_sum / arr_item_count) - self.flt_global_mean

        # mean-center the matrix (only observed entries)
        mat_centered = mat_dense.copy()
        for i in range(self.int_n_users):
            for j in np.where(mat_mask[i] > 0)[0]:
                mat_centered[i, j] -= (self.flt_global_mean + self.arr_user_bias[i] + self.arr_item_bias[j])

        # truncated SVD
        int_k_safe = min(self.int_k, min(mat_centered.shape) - 1)
        U, sigma, Vt = svds(csr_matrix(mat_centered), k=int_k_safe)

        # sort by descending singular values
        arr_idx = np.argsort(-sigma)
        self.U = U[:, arr_idx]
        self.arr_sigma = sigma[arr_idx]
        self.Vt = Vt[arr_idx, :]

        # precompute user latent factors (U * diag(sigma))
        self.mat_user_factors = self.U * self.arr_sigma[np.newaxis, :]

        # store set of seen items per user for recommendations
        self.dict_user_seen = df_train.groupby('user_id')['movie_id'].apply(set).to_dict()

        print(f'  SVD fit complete. Latent factors: {int_k_safe}')
        return self

    def predict(self, arr_user_ids, arr_movie_ids):
        arr_predictions = np.full(len(arr_user_ids), self.flt_global_mean)

        for i in range(len(arr_user_ids)):
            int_user_id = arr_user_ids[i]
            int_movie_id = arr_movie_ids[i]

            if int_user_id in self.dict_user_to_idx and int_movie_id in self.dict_item_to_idx:
                int_u_idx = self.dict_user_to_idx[int_user_id]
                int_m_idx = self.dict_item_to_idx[int_movie_id]

                flt_pred = (
                    self.flt_global_mean
                    + self.arr_user_bias[int_u_idx]
                    + self.arr_item_bias[int_m_idx]
                    + np.dot(self.mat_user_factors[int_u_idx], self.Vt[:, int_m_idx])
                )
                arr_predictions[i] = flt_pred
            elif int_user_id in self.dict_user_to_idx:
                int_u_idx = self.dict_user_to_idx[int_user_id]
                arr_predictions[i] = self.flt_global_mean + self.arr_user_bias[int_u_idx]
            elif int_movie_id in self.dict_item_to_idx:
                int_m_idx = self.dict_item_to_idx[int_movie_id]
                arr_predictions[i] = self.flt_global_mean + self.arr_item_bias[int_m_idx]

        return np.clip(arr_predictions, 1.0, 5.0)

    def recommend_top_k(self, int_user_id, int_k_items=10):
        if int_user_id not in self.dict_user_to_idx:
            return []

        int_u_idx = self.dict_user_to_idx[int_user_id]
        set_seen = self.dict_user_seen.get(int_user_id, set())

        # score all items
        arr_scores = (
            self.flt_global_mean
            + self.arr_user_bias[int_u_idx]
            + self.arr_item_bias
            + np.dot(self.mat_user_factors[int_u_idx], self.Vt)
        )

        # mask seen items
        for int_movie_id in set_seen:
            if int_movie_id in self.dict_item_to_idx:
                arr_scores[self.dict_item_to_idx[int_movie_id]] = -np.inf

        # get top-k indices
        arr_top_idx = np.argsort(-arr_scores)[:int_k_items]
        list_recs = [self.dict_idx_to_item[idx] for idx in arr_top_idx]
        return list_recs

In [ ]:
class RecommenderEvaluator:
    """Evaluator for recommender systems with rating and ranking metrics."""

    def __init__(self, int_k_at=10, flt_relevance_threshold=4.0):
        self.int_k_at = int_k_at
        self.flt_relevance_threshold = flt_relevance_threshold

    def compute_rating_metrics(self, arr_y_true, arr_y_pred):
        flt_rmse = np.sqrt(np.mean((arr_y_true - arr_y_pred) ** 2))
        flt_mae = np.mean(np.abs(arr_y_true - arr_y_pred))
        return {'flt_rmse': flt_rmse, 'flt_mae': flt_mae}

    def compute_ranking_metrics(self, df_test, cls_model):
        int_k = self.int_k_at
        flt_threshold = self.flt_relevance_threshold

        # get relevant items per user in test set
        df_relevant = df_test[df_test['rating'] >= flt_threshold]
        dict_user_relevant = df_relevant.groupby('user_id')['movie_id'].apply(set).to_dict()

        # only evaluate users who have at least 1 relevant item in test
        list_users = list(dict_user_relevant.keys())

        list_precision = []
        list_recall = []
        list_ndcg = []
        list_ap = []
        list_hit = []
        set_recommended_items = set()

        for int_user_id in tqdm(list_users, desc='Computing ranking metrics'):
            list_recs = cls_model.recommend_top_k(int_user_id, int_k_items=int_k)
            if len(list_recs) == 0:
                continue

            set_relevant = dict_user_relevant[int_user_id]
            set_recommended_items.update(list_recs)

            # precision@k
            int_hits = len(set(list_recs) & set_relevant)
            flt_precision = int_hits / int_k
            list_precision.append(flt_precision)

            # recall@k
            flt_recall = int_hits / len(set_relevant) if len(set_relevant) > 0 else 0.0
            list_recall.append(flt_recall)

            # ndcg@k
            arr_relevance = np.array([1.0 if item in set_relevant else 0.0 for item in list_recs])
            flt_dcg = np.sum(arr_relevance / np.log2(np.arange(2, len(arr_relevance) + 2)))
            arr_ideal = np.sort(arr_relevance)[::-1]
            flt_idcg = np.sum(arr_ideal / np.log2(np.arange(2, len(arr_ideal) + 2)))
            flt_ndcg = flt_dcg / flt_idcg if flt_idcg > 0 else 0.0
            list_ndcg.append(flt_ndcg)

            # average precision@k
            flt_ap = 0.0
            int_running_hits = 0
            for idx, item in enumerate(list_recs):
                if item in set_relevant:
                    int_running_hits += 1
                    flt_ap += int_running_hits / (idx + 1)
            flt_ap = flt_ap / min(int_k, len(set_relevant)) if len(set_relevant) > 0 else 0.0
            list_ap.append(flt_ap)

            # hit@k
            list_hit.append(1.0 if int_hits > 0 else 0.0)

        # catalog coverage
        int_total_items = len(cls_model.dict_item_to_idx)
        flt_coverage = len(set_recommended_items) / int_total_items if int_total_items > 0 else 0.0

        dict_metrics = {
            f'flt_precision_at_{int_k}': np.mean(list_precision),
            f'flt_recall_at_{int_k}': np.mean(list_recall),
            f'flt_ndcg_at_{int_k}': np.mean(list_ndcg),
            f'flt_map_at_{int_k}': np.mean(list_ap),
            f'flt_hit_rate_at_{int_k}': np.mean(list_hit),
            'flt_coverage': flt_coverage,
        }
        return dict_metrics

## Constants

In [ ]:
str_bucket = 'recommender-system-demo'
str_task = '03_svd'
str_dirname_output = './output'
str_uri = f's3://{str_bucket}/02_split_data/df.parquet'
int_k_at = 10
flt_relevance_threshold = 4.0

## Output Directory

In [ ]:
try:
    os.mkdir(str_dirname_output)
except:
    pass

## Import Data

In [ ]:
df = pd.read_parquet(str_uri)
print(f'Dataset: {df.shape[0]:,} rows, {df.shape[1]} columns')
print(f'Data sets: {df["data_set"].value_counts().to_dict()}')

## Train / Validation / Test Split

In [ ]:
df_train = df[df['data_set'] == 'train'].copy()
df_valid = df[df['data_set'] == 'valid'].copy()
df_test = df[df['data_set'] == 'test'].copy()

print(f'Train: {df_train.shape[0]:,} ratings')
print(f'Valid: {df_valid.shape[0]:,} ratings')
print(f'Test:  {df_test.shape[0]:,} ratings')

## Hyperparameter Tuning with Optuna

In [ ]:
dict_best_model = {'cls_model': None, 'flt_rmse': np.inf}

def objective(trial):
    int_k = trial.suggest_int('int_k', 10, 200)

    cls_model = SVDRecommender(int_k=int_k)
    cls_model.fit(df_train)

    arr_y_pred = cls_model.predict(
        df_valid['user_id'].values,
        df_valid['movie_id'].values,
    )
    flt_rmse = np.sqrt(np.mean((df_valid['rating'].values - arr_y_pred) ** 2))

    # track best model
    if flt_rmse < dict_best_model['flt_rmse']:
        dict_best_model['cls_model'] = cls_model
        dict_best_model['flt_rmse'] = flt_rmse

    return flt_rmse


int_n_trials = 30
pbar = tqdm(total=int_n_trials, desc='Optuna SVD')

def callback(study, trial):
    pbar.update(1)
    pbar.set_postfix({'best_rmse': f'{study.best_value:.4f}'})

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=int_n_trials, callbacks=[callback])
pbar.close()

print(f'\nBest hyperparameters: {study.best_params}')
print(f'Best validation RMSE: {study.best_value:.4f}')

## Train Final Model

In [ ]:
cls_model_inference = dict_best_model['cls_model']
print(f'Best model: k={cls_model_inference.int_k}')

## Rating Prediction Metrics

In [ ]:
cls_evaluator = RecommenderEvaluator(int_k_at=int_k_at, flt_relevance_threshold=flt_relevance_threshold)

# predictions
arr_y_hat_train = cls_model_inference.predict(df_train['user_id'].values, df_train['movie_id'].values)
arr_y_hat_valid = cls_model_inference.predict(df_valid['user_id'].values, df_valid['movie_id'].values)
arr_y_hat_test = cls_model_inference.predict(df_test['user_id'].values, df_test['movie_id'].values)

# rating metrics
dict_train_rating = cls_evaluator.compute_rating_metrics(df_train['rating'].values, arr_y_hat_train)
dict_valid_rating = cls_evaluator.compute_rating_metrics(df_valid['rating'].values, arr_y_hat_valid)
dict_test_rating = cls_evaluator.compute_rating_metrics(df_test['rating'].values, arr_y_hat_test)

print(f'Rating Metrics:')
print(f'  Train - RMSE: {dict_train_rating["flt_rmse"]:.4f}, MAE: {dict_train_rating["flt_mae"]:.4f}')
print(f'  Valid - RMSE: {dict_valid_rating["flt_rmse"]:.4f}, MAE: {dict_valid_rating["flt_mae"]:.4f}')
print(f'  Test  - RMSE: {dict_test_rating["flt_rmse"]:.4f}, MAE: {dict_test_rating["flt_mae"]:.4f}')

## Ranking Metrics

In [ ]:
print('Computing ranking metrics on validation set...')
dict_valid_ranking = cls_evaluator.compute_ranking_metrics(df_valid, cls_model_inference)
for key, val in dict_valid_ranking.items():
    print(f'  Valid {key}: {val:.4f}')

print('\nComputing ranking metrics on test set...')
dict_test_ranking = cls_evaluator.compute_ranking_metrics(df_test, cls_model_inference)
for key, val in dict_test_ranking.items():
    print(f'  Test  {key}: {val:.4f}')

## Save Tuning Results

In [ ]:
dict_tuning = {
    'str_model': 'SVD',
    'int_k': cls_model_inference.int_k,
    'flt_rmse_train': dict_train_rating['flt_rmse'],
    'flt_rmse_valid': dict_valid_rating['flt_rmse'],
    'flt_rmse_test': dict_test_rating['flt_rmse'],
    'flt_mae_train': dict_train_rating['flt_mae'],
    'flt_mae_valid': dict_valid_rating['flt_mae'],
    'flt_mae_test': dict_test_rating['flt_mae'],
}
dict_tuning.update({f'{k}_valid': v for k, v in dict_valid_ranking.items()})
dict_tuning.update({f'{k}_test': v for k, v in dict_test_ranking.items()})

df_tuning = pd.DataFrame([dict_tuning])
df_tuning.to_csv(f'{str_dirname_output}/df_tuning.csv', index=False)
print('Tuning results saved.')
print(df_tuning.T)

## Save Model

In [ ]:
joblib.dump(cls_model_inference, f'{str_dirname_output}/cls_model_inference.joblib')
print('Model saved.')

## Prediction Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('SVD: Prediction Analysis', fontsize=14, y=1.02)

# prediction distribution
axes[0].hist(arr_y_hat_test, bins=50, color='tab:blue', edgecolor='black', alpha=0.7, label='Predicted')
axes[0].hist(df_test['rating'].values, bins=5, color='tab:red', edgecolor='black', alpha=0.4, label='Actual')
axes[0].set_xlabel('Rating')
axes[0].set_ylabel('Count')
axes[0].set_title('Predicted vs Actual Rating Distribution (Test)')
axes[0].legend()

# residual distribution
arr_residuals = df_test['rating'].values - arr_y_hat_test
axes[1].hist(arr_residuals, bins=50, color='tab:orange', edgecolor='black', alpha=0.7)
axes[1].axvline(x=0, color='black', linestyle='--', linewidth=1)
axes[1].set_xlabel('Residual (Actual - Predicted)')
axes[1].set_ylabel('Count')
axes[1].set_title(f'Residual Distribution (Test) | Mean: {arr_residuals.mean():.3f}')

plt.tight_layout()
plt.savefig(f'{str_dirname_output}/prediction_distribution.png', bbox_inches='tight', dpi=150)
plt.show()

## Save Test Predictions

In [ ]:
df_test_preds = pd.DataFrame({
    'user_id': df_test['user_id'].values,
    'movie_id': df_test['movie_id'].values,
    'rating': df_test['rating'].values,
    'y_hat': arr_y_hat_test,
})
df_test_preds.to_parquet(f'{str_dirname_output}/test_predictions.parquet', index=False)
print(f'Test predictions saved: {df_test_preds.shape[0]:,} rows')